# H-N110 → H-N120 focal loss experiments

H-N110(Dice+Focal) 완료 후 H-N120(Tversky+Focal)을 자동 실행합니다. 두 실험 모두 기본 sampling을 사용합니다.


In [1]:
from pathlib import Path
import json, os, subprocess, sys, time
import pandas as pd
from IPython.display import clear_output, display

ROOT = Path("/home/stu03/workspace/model")
DATA_ROOT = Path("/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet")
PYTHON = ROOT / ".venv/bin/python"
BATCH_DIR = ROOT / "runs/batches/H-N110_H-N120"
BATCH_STATUS = BATCH_DIR / "batch_status.json"
assert PYTHON.is_file() and DATA_ROOT.is_dir()


## 1. 설정·테스트·plan 확인


In [2]:
subprocess.run([str(PYTHON), "-m", "pytest", "-q", "tests/test_focal_losses.py"], cwd=ROOT, check=True)
for name in ("H-N110", "H-N120"):
    subprocess.run([str(PYTHON), "-c", f"from brainon_ai.nnunet.config import load_config; load_config('configs/experiments/hemorrhage/{name}.yaml'); print('{name}: config OK')"], cwd=ROOT, check=True)
subprocess.run([str(PYTHON), "scripts/preprocessing/prepare_h_n110_h_n120_plans.py", "--preprocessed-root", str(DATA_ROOT / "nnUNet_preprocessed")], cwd=ROOT, check=True)
subprocess.run([str(PYTHON), "scripts/training/run_h_n110_h_n120.py", "--data-root", str(DATA_ROOT), "--dry-run", "--no-validation"], cwd=ROOT, check=True)


...                                                                      [100%]
3 passed in 0.68s
H-N110: config OK
H-N120: config OK
[ok] H-N110: /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed/Dataset001_ICHBinary/ICHPlansDiceFocalV1.json
[ok] H-N120: /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed/Dataset001_ICHBinary/ICHPlansTverskyFocalV1.json
/home/stu03/workspace/model/.venv/bin/python /home/stu03/workspace/model/scripts/preprocessing/prepare_h_n110_h_n120_plans.py --preprocessed-root /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed
H-N110: /home/stu03/workspace/model/.venv/bin/python -m brainon_ai.nnunet.runner --config /home/stu03/workspace/model/runs/batches/H-N110_H-N120/resolved/H-N110.yaml --data-root /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet --results-root /home/stu03/workspace/model/runs/nnunet --no-validation
H-N120: /home/stu03/workspace/mode

CompletedProcess(args=['/home/stu03/workspace/model/.venv/bin/python', 'scripts/training/run_h_n110_h_n120.py', '--data-root', '/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet', '--dry-run', '--no-validation'], returncode=0)

## 2. 두 실험 백그라운드 순차 실행\n\n처음 실행은 RESUME=False입니다. 중단된 checkpoint에서 이어갈 때만 True로 바꾸세요. 학습 중에는 full-volume validation을 생략하고, 두 학습이 끝난 뒤 동일한 평가 노트북으로 비교합니다.\n

In [3]:
RESUME = False
GPU = "0"
BATCH_DIR.mkdir(parents=True, exist_ok=True)
command = [str(PYTHON), "scripts/training/run_h_n110_h_n120.py", "--data-root", str(DATA_ROOT), "--gpu", GPU, "--no-validation"]
if RESUME:
    command.append("--resume")
launch_log = (BATCH_DIR / "launcher.log").open("a", encoding="utf-8")
process = subprocess.Popen(command, cwd=ROOT, stdout=launch_log, stderr=subprocess.STDOUT, start_new_session=True)
(BATCH_DIR / "launcher.pid").write_text(str(process.pid), encoding="utf-8")
print("PID:", process.pid)
print("log:", BATCH_DIR / "launcher.log")


PID: 144871
log: /home/stu03/workspace/model/runs/batches/H-N110_H-N120/launcher.log


## 3. 실시간 진행률


In [4]:
PROGRESS_FILES = {
    "H-N110": (
        ROOT
        / "runs/nnunet/Dataset001_ICHBinary/"
        "nnUNetTrainerICHBaseline__ICHPlansDiceFocalV1__3d_fullres/"
        "fold_0/progress.json"
    ),
    "H-N120": (
        ROOT
        / "runs/nnunet/Dataset001_ICHBinary/"
        "nnUNetTrainerICHBaseline__ICHPlansTverskyFocalV1__3d_fullres/"
        "fold_0/progress.json"
    ),
}


def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def fmt(seconds):
    if seconds is None:
        return "-"

    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


TERMINAL_STATUSES = {
    "completed",
    "early_stopped",
    "failed",
}

while True:
    clear_output(wait=True)

    batch = read_json(BATCH_STATUS)
    batch_items = batch.get("experiments", {})

    rows = []
    current_statuses = {}

    for name, path in PROGRESS_FILES.items():
        progress = read_json(path)
        batch_item = batch_items.get(name, {})

        status = progress.get(
            "status",
            batch_item.get("status", "waiting"),
        )
        current_statuses[name] = status

        rows.append({
            "실험": name,
            "상태": status,
            "Epoch": (
                f"{progress.get('current_epoch', 0)}"
                f"/{progress.get('total_epochs', '-')}"
            ),
            "진행률(%)": progress.get("progress_percent", 0),
            "ETA": fmt(progress.get("eta_seconds")),
            "Dice": progress.get("pseudo_dice"),
            "Recall": progress.get("pseudo_recall"),
            "Best Composite": progress.get("best_composite"),
        })

    # batch_status.json의 과거 failed 상태가 아니라
    # 각 실험의 최신 progress.json을 기준으로 전체 상태 계산
    running_experiment = next(
        (
            name
            for name, status in current_statuses.items()
            if status in {"running", "early_stopping"}
        ),
        None,
    )

    if running_experiment is not None:
        overall_status = "running"
    elif all(
        status in {"completed", "early_stopped"}
        for status in current_statuses.values()
    ):
        overall_status = "completed"
    elif any(
        status == "failed"
        for status in current_statuses.values()
    ):
        overall_status = "failed"
    else:
        overall_status = "waiting"

    print("전체:", overall_status)
    print("현재:", running_experiment or "-")
    print()

    display(pd.DataFrame(rows))

    # 두 실험 모두 실제 종료됐을 때만 반복 종료
    if all(
        status in TERMINAL_STATUSES
        for status in current_statuses.values()
    ):
        print("모든 실험이 종료됐습니다.")
        break

    time.sleep(10)

전체: completed
현재: -



,실험,상태,Epoch,진행률(%),ETA,Dice,Recall,Best Composite
0,H-N110,early_stopped,248/300,82.67,00:39:25,None,None,0.834635
1,H-N120,early_stopped,239/300,79.67,00:46:28,None,None,0.858636


모든 실험이 종료됐습니다.


In [5]:
actual_statuses = [row["상태"] for row in rows]

if any(status == "running" for status in actual_statuses):
    overall_status = "running"
elif all(status in {"completed", "early_stopped"} for status in actual_statuses):
    overall_status = "completed"
else:
    overall_status = batch.get("status", "starting")

print("전체:", overall_status)
print("현재:", next(
    (row["실험"] for row in rows if row["상태"] == "running"),
    "-"
))

전체: completed
현재: -


In [6]:
from pathlib import Path
import json
import subprocess

import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path("/home/stu03/workspace/model")
DATA_ROOT = Path(
    "/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet"
)
RESULTS_ROOT = ROOT / "runs/nnunet"

OUTPUT_DIR = ROOT / "runs/comparisons/H-N110_H-N120_full_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {
    "H-N110": {
        "dataset": "Dataset001_ICHBinary",
        "plans": "ICHPlansDiceFocalV1",
    },
    "H-N120": {
        "dataset": "Dataset001_ICHBinary",
        "plans": "ICHPlansTverskyFocalV1",
    },
}

SIZE_ORDER = [
    "<1 mL",
    "1–5 mL",
    "5–20 mL",
    "≥20 mL",
]

In [7]:
command = [
    str(ROOT / ".venv/bin/python"),
    "scripts/validation/complete_h_n110_h_n120_validation.py",
]

subprocess.run(
    command,
    cwd=ROOT,
    check=True,
)

===== H-N110 full-volume validation =====
Trainer 'nnUNetTrainerICHBaseline' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Searching for class nnUNetTrainerICHBaseline in folder /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers with current module None
  Inspecting module: configurable_trainer
Found class nnUNetTrainerICHBaseline in configurable_trainer
Using trainer 'nnUNetTrainerICHBaseline' from: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
####################

CompletedProcess(args=['/home/stu03/workspace/model/.venv/bin/python', 'scripts/validation/complete_h_n110_h_n120_validation.py'], returncode=0)

In [8]:
registry = {}

for experiment, info in EXPERIMENTS.items():
    fold_dir = (
        RESULTS_ROOT
        / info["dataset"]
        / (
            "nnUNetTrainerICHBaseline"
            f"__{info['plans']}"
            "__3d_fullres"
        )
        / "fold_0"
    )

    summary_path = fold_dir / "validation/summary.json"
    checkpoint_path = fold_dir / "checkpoint_best_composite.pth"

    registry[experiment] = {
        **info,
        "fold_dir": fold_dir,
        "summary_path": summary_path,
    }

    print(
        experiment,
        "\n checkpoint:", checkpoint_path.is_file(),
        "\n summary:", summary_path.is_file(),
        "\n path:", summary_path,
        "\n",
    )

missing = [
    experiment
    for experiment, info in registry.items()
    if not info["summary_path"].is_file()
]

if missing:
    raise FileNotFoundError(
        f"full-volume validation 결과가 없습니다: {missing}"
    )

H-N110 
 checkpoint: True 
 summary: True 
 path: /home/stu03/workspace/model/runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansDiceFocalV1__3d_fullres/fold_0/validation/summary.json 

H-N120 
 checkpoint: True 
 summary: True 
 path: /home/stu03/workspace/model/runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansTverskyFocalV1__3d_fullres/fold_0/validation/summary.json 



In [9]:
def get_case_id(path):
    filename = str(path).replace("\\", "/").split("/")[-1]

    if filename.endswith(".nii.gz"):
        return filename[:-7]

    return Path(filename).stem


def get_volume_group(volume_ml):
    if volume_ml < 1:
        return "<1 mL"
    if volume_ml < 5:
        return "1–5 mL"
    if volume_ml < 20:
        return "5–20 mL"
    return "≥20 mL"


rows = []

for experiment, info in registry.items():
    summary = json.loads(
        info["summary_path"].read_text(encoding="utf-8")
    )

    gt_dir = (
        DATA_ROOT
        / "nnUNet_preprocessed"
        / info["dataset"]
        / "gt_segmentations"
    )

    for case_result in summary["metric_per_case"]:
        case_id = get_case_id(case_result["prediction_file"])
        metrics = case_result["metrics"]["1"]

        tp = float(metrics["TP"])
        fp = float(metrics["FP"])
        fn = float(metrics["FN"])
        reference_voxels = float(metrics["n_ref"])

        gt_path = gt_dir / f"{case_id}.nii.gz"
        spacing = nib.load(gt_path).header.get_zooms()[:3]
        voxel_volume_ml = float(np.prod(spacing)) / 1000.0
        hemorrhage_volume_ml = reference_voxels * voxel_volume_ml

        positive = reference_voxels > 0
        predicted_voxels = tp + fp

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else np.nan
        )
        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else np.nan
        )

        rows.append({
            "experiment": experiment,
            "case_id": case_id,
            "source": (
                "PhysioNet"
                if case_id.startswith("PHYS_")
                else "BHSD"
            ),
            "positive": positive,
            "hemorrhage_volume_ml": hemorrhage_volume_ml,
            "volume_group": (
                get_volume_group(hemorrhage_volume_ml)
                if positive
                else "negative"
            ),
            "dice": float(metrics["Dice"]),
            "recall": recall,
            "precision": precision,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "predicted_voxels": predicted_voxels,
            "detected": tp > 0,
            "true_negative": (
                not positive and predicted_voxels == 0
            ),
        })


case_metrics = pd.DataFrame(rows)

case_metrics["volume_group"] = pd.Categorical(
    case_metrics["volume_group"],
    categories=SIZE_ORDER + ["negative"],
    ordered=True,
)

display(case_metrics.head())

case_metrics.to_csv(
    OUTPUT_DIR / "per_case_metrics.csv",
    index=False,
)

,experiment,case_id,source,positive,hemorrhage_volume_ml,volume_group,dice,recall,precision,tp,fp,fn,predicted_voxels,detected,true_negative
0,H-N110,BHSD_ID_02b882cc_ID_a4892e60ae,BHSD,True,105.491404,≥20 mL,0.860270,0.941145,0.792195,77220.0,20256.0,4829.0,97476.0,True,False
1,H-N110,BHSD_ID_07365cde_ID_8e661b8f07,BHSD,True,9.549252,5–20 mL,0.934658,0.970484,0.901383,7102.0,777.0,216.0,7879.0,True,False
2,H-N110,BHSD_ID_0b10cbee_ID_f91d6a7cd2,BHSD,True,2.676365,1–5 mL,0.000000,0.000000,0.000000,0.0,530.0,2147.0,530.0,False,False
3,H-N110,BHSD_ID_10c64d22_ID_4c9aad31ca,BHSD,True,120.548283,≥20 mL,0.839440,0.888166,0.795782,82365.0,21137.0,10371.0,103502.0,True,False
4,H-N110,BHSD_ID_1cbe2853_ID_37afefdc7c,BHSD,True,21.587487,≥20 mL,0.911404,0.965087,0.863379,16475.0,2607.0,596.0,19082.0,True,False


In [10]:
def summarize_positive(data, experiment, scope):
    return {
        "experiment": experiment,
        "scope": scope,
        "cases": len(data),
        "mean_dice": data["dice"].mean(),
        "median_dice": data["dice"].median(),
        "mean_recall": data["recall"].mean(),
        "mean_precision": data["precision"].mean(),
        "detection_rate": data["detected"].mean(),
        "detected_cases": int(data["detected"].sum()),
        "zero_dice_cases": int(
            data["dice"].fillna(0).eq(0).sum()
        ),
    }


overall_rows = []

for experiment in EXPERIMENTS:
    positive_cases = case_metrics[
        (case_metrics["experiment"] == experiment)
        & case_metrics["positive"]
    ]

    overall_rows.append(
        summarize_positive(
            positive_cases,
            experiment,
            "positive_all",
        )
    )


overall_summary = pd.DataFrame(overall_rows)

display(
    overall_summary.style.format({
        "mean_dice": "{:.4f}",
        "median_dice": "{:.4f}",
        "mean_recall": "{:.4f}",
        "mean_precision": "{:.4f}",
        "detection_rate": "{:.2%}",
    })
)

overall_summary.to_csv(
    OUTPUT_DIR / "overall_summary.csv",
    index=False,
)

,experiment,scope,cases,mean_dice,median_dice,mean_recall,mean_precision,detection_rate,detected_cases,zero_dice_cases
0,H-N110,positive_all,35,0.5999,0.6489,0.6482,0.6080,91.43%,32,3
1,H-N120,positive_all,35,0.6002,0.7260,0.6315,0.6245,91.43%,32,3


In [11]:
size_rows = []

for experiment in EXPERIMENTS:
    positive_cases = case_metrics[
        (case_metrics["experiment"] == experiment)
        & case_metrics["positive"]
    ]

    for volume_group in SIZE_ORDER:
        group_cases = positive_cases[
            positive_cases["volume_group"] == volume_group
        ]

        if len(group_cases) == 0:
            continue

        size_rows.append(
            summarize_positive(
                group_cases,
                experiment,
                volume_group,
            )
        )


size_summary = pd.DataFrame(size_rows)

display(
    size_summary.style.format({
        "mean_dice": "{:.4f}",
        "median_dice": "{:.4f}",
        "mean_recall": "{:.4f}",
        "mean_precision": "{:.4f}",
        "detection_rate": "{:.2%}",
    })
)

size_summary.to_csv(
    OUTPUT_DIR / "size_summary.csv",
    index=False,
)

,experiment,scope,cases,mean_dice,median_dice,mean_recall,mean_precision,detection_rate,detected_cases,zero_dice_cases
0,H-N110,<1 mL,3,0.0501,0.0706,0.5238,0.0264,66.67%,2,1
1,H-N110,1–5 mL,8,0.4420,0.5060,0.4518,0.4540,75.00%,6,2
2,H-N110,5–20 mL,8,0.5769,0.5653,0.5614,0.6144,100.00%,8,0
3,H-N110,≥20 mL,16,0.7935,0.8614,0.8131,0.7907,100.00%,16,0
4,H-N120,<1 mL,3,0.1180,0.0522,0.4372,0.0759,66.67%,2,1
5,H-N120,1–5 mL,8,0.4271,0.4910,0.4248,0.4675,75.00%,6,2
6,H-N120,5–20 mL,8,0.5771,0.5709,0.5588,0.6576,100.00%,8,0
7,H-N120,≥20 mL,16,0.7886,0.8455,0.8075,0.7893,100.00%,16,0


In [12]:
small_rows = []

for experiment in EXPERIMENTS:
    small_cases = case_metrics[
        (case_metrics["experiment"] == experiment)
        & case_metrics["positive"]
        & (case_metrics["hemorrhage_volume_ml"] < 5)
    ]

    small_rows.append(
        summarize_positive(
            small_cases,
            experiment,
            "<5 mL",
        )
    )


small_summary = pd.DataFrame(small_rows)

display(
    small_summary.style.format({
        "mean_dice": "{:.4f}",
        "median_dice": "{:.4f}",
        "mean_recall": "{:.4f}",
        "mean_precision": "{:.4f}",
        "detection_rate": "{:.2%}",
    })
)

small_summary.to_csv(
    OUTPUT_DIR / "small_under_5ml_summary.csv",
    index=False,
)

,experiment,scope,cases,mean_dice,median_dice,mean_recall,mean_precision,detection_rate,detected_cases,zero_dice_cases
0,H-N110,<5 mL,11,0.3351,0.1988,0.4714,0.3374,72.73%,8,3
1,H-N120,<5 mL,11,0.3428,0.3017,0.4282,0.3607,72.73%,8,3


In [13]:
negative_rows = []

for experiment in EXPERIMENTS:
    negative_cases = case_metrics[
        (case_metrics["experiment"] == experiment)
        & ~case_metrics["positive"]
    ]

    negative_rows.append({
        "experiment": experiment,
        "negative_cases": len(negative_cases),
        "true_negative_cases": int(
            negative_cases["true_negative"].sum()
        ),
        "case_specificity": (
            negative_cases["true_negative"].mean()
        ),
        "false_positive_cases": int(
            (~negative_cases["true_negative"]).sum()
        ),
        "mean_fp_voxels": negative_cases["fp"].mean(),
        "median_fp_voxels": negative_cases["fp"].median(),
    })


negative_summary = pd.DataFrame(negative_rows)

display(
    negative_summary.style.format({
        "case_specificity": "{:.2%}",
        "mean_fp_voxels": "{:.1f}",
        "median_fp_voxels": "{:.1f}",
    })
)

negative_summary.to_csv(
    OUTPUT_DIR / "negative_summary.csv",
    index=False,
)

,experiment,negative_cases,true_negative_cases,case_specificity,false_positive_cases,mean_fp_voxels,median_fp_voxels
0,H-N110,5,1,20.00%,4,300.2,151.0
1,H-N120,5,1,20.00%,4,286.6,276.0


In [14]:
leaderboard = (
    overall_summary[
        [
            "experiment",
            "cases",
            "mean_dice",
            "mean_recall",
            "mean_precision",
            "detection_rate",
            "zero_dice_cases",
        ]
    ]
    .merge(
        small_summary[
            [
                "experiment",
                "mean_dice",
                "mean_recall",
                "detection_rate",
            ]
        ],
        on="experiment",
        suffixes=("_all_positive", "_under_5ml"),
    )
    .merge(
        negative_summary[
            [
                "experiment",
                "case_specificity",
                "mean_fp_voxels",
            ]
        ],
        on="experiment",
    )
    .sort_values(
        "mean_dice_all_positive",
        ascending=False,
    )
)


display(
    leaderboard.style.format({
        "mean_dice_all_positive": "{:.4f}",
        "mean_recall_all_positive": "{:.4f}",
        "mean_precision": "{:.4f}",
        "detection_rate_all_positive": "{:.2%}",
        "mean_dice_under_5ml": "{:.4f}",
        "mean_recall_under_5ml": "{:.4f}",
        "detection_rate_under_5ml": "{:.2%}",
        "case_specificity": "{:.2%}",
        "mean_fp_voxels": "{:.1f}",
    })
)

leaderboard.to_csv(
    OUTPUT_DIR / "leaderboard.csv",
    index=False,
)

print("저장 위치:", OUTPUT_DIR)

,experiment,cases,mean_dice_all_positive,mean_recall_all_positive,mean_precision,detection_rate_all_positive,zero_dice_cases,mean_dice_under_5ml,mean_recall_under_5ml,detection_rate_under_5ml,case_specificity,mean_fp_voxels
1,H-N120,35,0.6002,0.6315,0.6245,91.43%,3,0.3428,0.4282,72.73%,20.00%,286.6
0,H-N110,35,0.5999,0.6482,0.6080,91.43%,3,0.3351,0.4714,72.73%,20.00%,300.2


저장 위치: /home/stu03/workspace/model/runs/comparisons/H-N110_H-N120_full_validation
